In [ ]:
import os
import zipfile
import duckdb
import pyarrow
import pyarrow.parquet as pq
import pandas as pd
from google.colab import drive

#functions defined at top
def extract_zip(zip_path, target_dir):
    print(f"unzipping dataset to local ssd: {os.path.basename(zip_path)}")
    with zipfile.ZipFile(zip_path, 'r') as z:
        extracted_fname = z.namelist()[0]
        z.extractall(target_dir)
    extracted_path = os.path.join(target_dir, extracted_fname)
    print(f"saved uncompressed: {extracted_fname}")
    return extracted_path

def convert_stata_to_parquet(dta_path, parquet_path, chunksize=5000000):
    print("converting stata file to fast local parquet")
    if os.path.exists(parquet_path):
        os.remove(parquet_path)
    reader = pd.read_stata(dta_path, chunksize=chunksize)
    writer = None
    for i, chunk in enumerate(reader):
        chunk['pat_no'] = chunk['pat_no'].astype(str)
        table = pyarrow.Table.from_pandas(chunk)
        if writer is None:
            writer = pq.ParquetWriter(parquet_path, table.schema)
        writer.write_table(table)
        print(f"converted {(i + 1) * (chunksize // 1000000):,} million rows to disk")
    if writer is not None:
        writer.close()

def print_table_count(con, table_name, label):
    count = con.execute(f"SELECT COUNT(*) FROM {table_name};").fetchone()[0]
    print(f"[diagnostic] {label} ({table_name}): {count:,} rows")

#setup and paths
print("step 2: duckdb parquet conversion")

drive_mount_point = os.path.join(os.sep, "content", "drive")
drive.mount(drive_mount_point, force_remount=False)

drive_data_dir = os.path.join(drive_mount_point, "MyDrive", "data")
ssd_dir = os.path.join(os.sep, "tmp", "uspto_uncompressed")
os.makedirs(ssd_dir, exist_ok=True)

step1_parquet = os.path.join(drive_data_dir, "step1_metrics.parquet")
claims_zip_path = os.path.join(drive_data_dir, "patent_claims_stats.dta.zip")

#init duckdb
db_path = os.path.join(ssd_dir, "step2_temp.db")
if os.path.exists(db_path):
    os.remove(db_path)

con = duckdb.connect(db_path)
con.execute("SET memory_limit = '4GB';")
con.execute("SET threads = 2;")

#unzip and prepare lookup keys
local_dta_path = extract_zip(claims_zip_path, ssd_dir)

print("building fast target key index")
con.execute(f"""
    CREATE OR REPLACE TABLE target_keys AS
    SELECT DISTINCT
        CAST(patent_id AS VARCHAR) AS original_patent_id,
        REGEXP_REPLACE(LTRIM(REGEXP_REPLACE(CAST(patent_id AS VARCHAR), '^0+', ''), '0'), '[^0-9]', '', 'g') AS clean_key
    FROM '{step1_parquet}';
""")

print_table_count(con, "target_keys", "pre-merge unique target keys")

#convert stata to local parquet via pyarrow
local_claims_parquet = os.path.join(ssd_dir, "claims_raw.parquet")
convert_stata_to_parquet(local_dta_path, local_claims_parquet)

os.remove(local_dta_path)

#pre-clean claims keys and inner join
print("executing pre-indexed high-speed join")
con.execute(f"""
    CREATE OR REPLACE TABLE matched_claims AS
    WITH cleaned_claims AS (
        SELECT
            *,
            REGEXP_REPLACE(LTRIM(TRIM(pat_no), '0'), '[^0-9]', '', 'g') AS clean_key
        FROM '{local_claims_parquet}'
    )
    SELECT
        t.original_patent_id AS patent_id,
        c.* EXCLUDE (pat_no, clean_key)
    FROM cleaned_claims c
    INNER JOIN target_keys t ON c.clean_key = t.clean_key;
""")

print_table_count(con, "matched_claims", "post-merge matched claims")
total_matches = con.execute("SELECT COUNT(*) FROM matched_claims;").fetchone()[0]

#export to drive and clean ssd
if total_matches > 0:
    step2_parquet = os.path.join(drive_data_dir, "step2_claims.parquet")
    con.execute(f"COPY matched_claims TO '{step2_parquet}' (FORMAT PARQUET);")
    print(f"saved to drive: {step2_parquet}")

if os.path.exists(local_claims_parquet):
    os.remove(local_claims_parquet)

#setup and paths
print("step 3: sector normalization and master panel assembly")

if not os.path.exists(step1_parquet) or not os.path.exists(step2_parquet):
    raise FileNotFoundError("missing intermediate files")

con = duckdb.connect()
con.execute("SET threads = 2;")

print("loading datasets into duckdb")
con.execute(f"""
    CREATE OR REPLACE TABLE step1_data AS SELECT * FROM '{step1_parquet}';
    CREATE OR REPLACE TABLE step2_data AS SELECT * FROM '{step2_parquet}';
""")

print_table_count(con, "step1_data", "pre-merge base step1 cohort")
print_table_count(con, "step2_data", "pre-merge raw step2 claims")

#aggregate patent claims
print("aggregating claims to patent-level metrics")
con.execute("""
    CREATE OR REPLACE TABLE claims_aggregated AS
    SELECT
        patent_id,
        COUNT(claim_no) AS total_claims,
        AVG(word_ct) AS avg_words_per_claim,
        SUM(word_ct) AS total_claim_words,
        MAX(CASE WHEN ind_flg = '1' THEN 1 ELSE 0 END) AS has_independent_claims
    FROM step2_data
    GROUP BY patent_id;
""")

print_table_count(con, "claims_aggregated", "aggregated patent-level claims")

#cpc subclass x grant year benchmarking
print("computing cpc subclass x grant year benchmarks")
con.execute("""
    CREATE OR REPLACE TABLE cpc_year_benchmarks AS
    SELECT
        cpc_subclass,
        grant_year,
        AVG(raw_forward_citations) AS mean_cell_forward_cites,
        COUNT(*) AS cell_patent_count
    FROM step1_data
    GROUP BY cpc_subclass, grant_year;

    CREATE OR REPLACE TABLE master_patent_panel AS
    SELECT
        s1.*,
        CAST(s1.npl_citations AS FLOAT) / NULLIF(s1.backward_pat_citations + s1.npl_citations, 0) AS npl_ratio,
        s1.raw_forward_citations / NULLIF(bm.mean_cell_forward_cites, 0) AS normalized_forward_citations,
        bm.mean_cell_forward_cites AS peer_benchmark_mean_cites,
        COALESCE(cl.total_claims, 0) AS total_claims,
        COALESCE(cl.avg_words_per_claim, 0) AS avg_words_per_claim,
        COALESCE(cl.total_claim_words, 0) AS total_claim_words,
        COALESCE(cl.has_independent_claims, 0) AS has_independent_claims
    FROM step1_data s1
    LEFT JOIN cpc_year_benchmarks bm
        ON s1.cpc_subclass = bm.cpc_subclass AND s1.grant_year = bm.grant_year
    LEFT JOIN claims_aggregated cl
        ON s1.patent_id = cl.patent_id;
""")

print_table_count(con, "master_patent_panel", "post-merge master panel assembly")

#export master panel
output_csv_path = os.path.join(drive_data_dir, "cleaned_patent_panel.csv")
output_parquet_path = os.path.join(drive_data_dir, "cleaned_patent_panel.parquet")

print("exporting master panel to drive")
con.execute(f"COPY master_patent_panel TO '{output_csv_path}' (HEADER, DELIMITER ',');")
con.execute(f"COPY master_patent_panel TO '{output_parquet_path}' (FORMAT PARQUET);")

#clean up intermediate files
if os.path.exists(step1_parquet):
    os.remove(step1_parquet)
if os.path.exists(step2_parquet):
    os.remove(step2_parquet)

final_count = con.execute("SELECT COUNT(*) FROM master_patent_panel;").fetchone()[0]

print("phase 1 complete")
print(f"master cohort patents: {final_count:,}")
print(f"saved parquet: {output_parquet_path}")
print(f"saved csv: {output_csv_path}")